# Configuration System Demonstration

This notebook demonstrates the configuration-driven architecture of the multicolor nanoparticle classification pipeline. It showcases how the system manages experimental metadata, validates configurations, and organizes data paths.

## Overview

The pipeline uses a **3-tier YAML configuration system**:

1. `class_definitions.yaml`: Database of nanoparticle classes with elemental composition fractions
2. `config.yaml`: Reusable templates for data processing and path generation
3. `experiments.yaml`: Registry of all experiments with metadata and configuration references

The `config_loader` module provides a unified interface to:
- Validate all configurations using Pydantic schemas
- Cross-reference data between files
- Merge configurations for specific experiments
- Generate file paths automatically

---

## 1. Setup and Imports

In [1]:
# Standard imports
import sys
from pathlib import Path
from pprint import pprint

# Define PROJECT_ROOT
if Path.cwd().name == 'notebooks':
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

# Ensure PROJECT_ROOT is in sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Project imports
from src.data import config_loader
print("✅ Import successful")

✅ Import successful


## 2. Comprehensive Configuration Validation

The system validates all three YAML files and cross-references between them. This ensures, for instance:
- All class labels referenced in `experiments.yaml` exist in `class_definitions.yaml`
- All config templates referenced in `experiments.yaml` exist in `config.yaml`
- All elemental fractions in `class_definitions.yaml` sum to 1.0
- All mixture priors in `experiments.yaml` sum to 1.0

**Note:** We validate once here and reuse the validated objects throughout the notebook to avoid redundant validation.

**Note:** We use `verbose=True` here for educational purposes. In production scripts (like `main.py`), use `verbose=False` 
and provide custom summaries.

In [2]:
# Validate all configuration files with cross-reference checking (run once)
# Note: validated_configs = class_defs, pipeline_config, experiments_db
validated_configs = config_loader.verify_experiment_references(verbose=True)


COMPREHENSIVE VALIDATION: All config files + cross-references

Verifying class definitions...
✅ Class definitions (class_definitions.yaml) verified successfully.
Verifying pipeline configuration...
✅ Pipeline configuration (config.yaml) verified successfully.
Verifying experiments database structure...
✅ Experiments database (experiments.yaml) verified successfully.

Cross-validating references between config files...
  → Validating training experiment references...
  → Validating inference experiment references...

✅ COMPREHENSIVE VALIDATION COMPLETE



## 3. Explore Available Experiments

The system organizes experiments by:
- **Dataset type**: Training data (known compositions) vs. Inference data (mixture compositions to determine)
- **Group**: Physical measurements vs. simulations, single particles vs. multi-particle mixtures

In [3]:
# Displays all available experiments organized by dataset type and group
print("Available Experiments:")
print(config_loader.format_experiments_summary())

Available Experiments:

📁 Training Data:
   └─ physical_single_particle:
      • 2024-06-29_Dy100
      • 2024-06-29_Ho100
      • 2024-07-01_Dy100

📁 Inference Data:
   └─ physical_multi_particle:
      • 2025-07-02_6-component-mixture

📊 Summary: 3 training experiment(s), 1 inference experiment(s)


## 4. Load and Merge Configuration for a Training Experiment

For each experiment, the system:
1. Loads experiment-specific metadata from `experiments.yaml`
2. Merges in class definitions from `class_definitions.yaml`
3. Merges in processing templates from `config.yaml`
4. Returns a complete, ready-to-use configuration dictionary

In [4]:
# Select a training experiment to demonstrate
exp_id = '2024-06-29_Dy100'

# Load the merged configuration (uses already-validated configs)
config, dataset_type, group_name = config_loader.get_full_experiment_config(
    exp_id, validated_configs=validated_configs
)

print(f"Experiment: {exp_id}")
print(f"Dataset type: {dataset_type}")
print(f"Group: {group_name}\n")

print("Key configuration values:")
print(f"  Composition: {config['composition_label']}")
print(f"  Elemental fractions: {config['elemental_fractions']}")
print(f"  Host matrix: {config['host_matrix']}")
print(f"  Beam current: {config['beam_current']['estimate']} {config['beam_current']['unit']}")

Experiment: 2024-06-29_Dy100
Dataset type: training_data
Group: physical_single_particle

Key configuration values:
  Composition: Dy100
  Elemental fractions: {'Tb': 0.0, 'Dy': 1.0, 'Ho': 0.0, 'Gd': 0.0, 'Y': 0.0}
  Host matrix: Sodium Fluoride (NaXF4, X=rare earth element(s))
  Beam current: 0.175 nA


### Inspect Merged Raw Data Configuration

The `raw_config` contains all channel definitions and filename prefixes:

In [5]:
print("Raw data acquisition channels:\n")
pprint(config['raw_config']['acquisitions'])

Raw data acquisition channels:

{'CL_SE': {'CL': {'blue': {'source_prefix': '593CP_NP_', 'target_dopant': 'Tb'},
                  'green': {'source_prefix': '593SP_NP_',
                            'target_dopant': 'Dy'},
                  'red': {'source_prefix': '593LP_NP_', 'target_dopant': 'Ho'}},
           'SE': {'source_prefix': 'SE2_NP_', 'target_dopant': None}},
 'SEM': {'source_prefix': 'NP_', 'target_dopant': None}}


### Inspect Merged Processed Data Configuration

The `processed_config` defines output specifications for each processing method:

In [6]:
print("Available processing methods:")
print(list(config['processed_config'].keys()))

print("\nSummed images output configuration:\n")
pprint(config['processed_config']['summed_images']['outputs'])

Available processing methods:
['summed_images']

Summed images output configuration:

{'CL_SE': {'CL': {'blue': {'filename_prefix': 'summed_blue'},
                  'green': {'filename_prefix': 'summed_green'},
                  'red': {'filename_prefix': 'summed_red'}},
           'SE': {'filename_prefix': 'summed_SE2'}}}


## 5. Load and Merge Configuration for an Inference Experiment

Inference experiments differ from training experiments:
- They have `possible_composition_labels` instead of a single `composition_label` (as in the current single-composition physical datasets)
- They include `mixture_priors` (Bayesian priors for mixture components)
- The system includes `class_definitions` for all possible compositions

In [7]:
# Select an inference experiment
infer_exp_id = '2025-07-02_6-component-mixture'

# Load the merged configuration (uses already-validated configs)
infer_config, infer_dataset, infer_group = config_loader.get_full_experiment_config(
    infer_exp_id, validated_configs=validated_configs
)

print(f"Inference Experiment: {infer_exp_id}")
print(f"Dataset type: {infer_dataset}")
print(f"Group: {infer_group}\n")

print("Mixture information:")
print(f"  Possible compositions: {infer_config['possible_composition_labels']}")
print(f"  Mixture priors: {infer_config['mixture_priors']}")

print("\nClass definitions for each possible composition:")
for label in infer_config['possible_composition_labels']:
    print(f"  {label}: {infer_config['class_definitions'][label]['elemental_fractions']}")

Inference Experiment: 2025-07-02_6-component-mixture
Dataset type: inference_data
Group: physical_multi_particle

Mixture information:
  Possible compositions: ['Dy100', 'Ho100', 'Dy80Ho20', 'Dy65Ho35', 'Dy50Ho50', 'Dy30Ho70']
  Mixture priors: {'Dy100': 0.16666666666666666, 'Ho100': 0.16666666666666666, 'Dy80Ho20': 0.16666666666666666, 'Dy65Ho35': 0.16666666666666666, 'Dy50Ho50': 0.16666666666666666, 'Dy30Ho70': 0.16666666666666666}

Class definitions for each possible composition:
  Dy100: {'Tb': 0.0, 'Dy': 1.0, 'Ho': 0.0, 'Gd': 0.0, 'Y': 0.0}
  Ho100: {'Tb': 0.0, 'Dy': 0.0, 'Ho': 1.0, 'Gd': 0.0, 'Y': 0.0}
  Dy80Ho20: {'Tb': 0.0, 'Dy': 0.8, 'Ho': 0.2, 'Gd': 0.0, 'Y': 0.0}
  Dy65Ho35: {'Tb': 0.0, 'Dy': 0.65, 'Ho': 0.35, 'Gd': 0.0, 'Y': 0.0}
  Dy50Ho50: {'Tb': 0.0, 'Dy': 0.5, 'Ho': 0.5, 'Gd': 0.0, 'Y': 0.0}
  Dy30Ho70: {'Tb': 0.0, 'Dy': 0.3, 'Ho': 0.7, 'Gd': 0.0, 'Y': 0.0}


## 6. Automatic Path Generation

The system automatically generates directory paths following a consistent hierarchy:

```
data/{dataset_type}/{group_component_1}/{group_component_2}/{experiment_id}/
├── raw/              # Raw microscopy images
├── processed/        # Processed outputs
└── metadata/         # Experimental metadata (CSV files, etc.)
```

In [8]:
# Generate paths for the training experiment (uses already-validated configs)
paths = config_loader.get_experiment_paths(
    exp_id, validated_configs=validated_configs
)

print(f"Directory structure for experiment '{exp_id}':\n")
print(f"Base directory:   ./{Path(paths['base']).relative_to(PROJECT_ROOT)}\n")

print("Subdirectories:")
print(f"  Raw data:         ./{Path(paths['raw']).relative_to(PROJECT_ROOT)}")
print(f"  Processed data:   ./{Path(paths['processed']).relative_to(PROJECT_ROOT)}")
print(f"  Metadata:         ./{Path(paths['metadata']).relative_to(PROJECT_ROOT)}")

Directory structure for experiment '2024-06-29_Dy100':

Base directory:   ./data/training/physical/single_particle/2024-06-29_Dy100

Subdirectories:
  Raw data:         ./data/training/physical/single_particle/2024-06-29_Dy100/raw
  Processed data:   ./data/training/physical/single_particle/2024-06-29_Dy100/processed
  Metadata:         ./data/training/physical/single_particle/2024-06-29_Dy100/metadata


### Metadata File Paths

Metadata filenames use template substitution with the experiment ID:

In [9]:
# Get metadata file path (template: '{id}_regions.csv')
regions_path = config_loader.get_metadata_filepath(
    exp_id, 
    metadata_type='regions',
    validated_configs=validated_configs
)

print(
    "Regions metadata file:\n",
    f" ./{Path(regions_path).relative_to(PROJECT_ROOT)}"
)

Regions metadata file:
  ./data/training/physical/single_particle/2024-06-29_Dy100/metadata/2024-06-29_Dy100_regions.csv


## 7. Raw Data File Path Patterns

For each region, the system generates glob patterns for all raw data channels.
The nested structure follows the acquisition hierarchy:

```
raw/
├── {region_id}/
│   ├── CL_SE/
│   │   ├── CL/
│   │   │   ├── blue/    # Blue channel CL images
│   │   │   ├── green/   # Green channel CL images
│   │   │   └── red/     # Red channel CL images
│   │   └── SE/          # Secondary electron images
│   └── SEM/             # SEM images
```

In [10]:
# Get raw file path patterns for region_1 (uses already-validated configs)
region_id = 'region_1'
raw_paths = config_loader.get_raw_filepaths(
    exp_id, 
    region_id=region_id,
    validated_configs=validated_configs
)

print(f"Raw data file patterns for {region_id}:\n")
for channel, pattern in raw_paths.items():
    print(f"  {channel:12s}: ./{Path(pattern).relative_to(PROJECT_ROOT)}")

Raw data file patterns for region_1:

  CL_blue     : ./data/training/physical/single_particle/2024-06-29_Dy100/raw/region_1/CL_SE/CL/blue/593CP_NP_*
  CL_green    : ./data/training/physical/single_particle/2024-06-29_Dy100/raw/region_1/CL_SE/CL/green/593SP_NP_*
  CL_red      : ./data/training/physical/single_particle/2024-06-29_Dy100/raw/region_1/CL_SE/CL/red/593LP_NP_*
  SE          : ./data/training/physical/single_particle/2024-06-29_Dy100/raw/region_1/CL_SE/SE/SE2_NP_*
  SEM         : ./data/training/physical/single_particle/2024-06-29_Dy100/raw/region_1/SEM/NP_*


### Check for Actual Data Files

Currently, this is a demonstration of the configuration system, and raw data files may not yet be present in the repository. Let's check if any actual data exists using the glob patterns we just generated:

In [11]:
import glob

# Check if any raw data files exist for the CL_red channel
CL_red_pattern = raw_paths['CL_red']
matching_files = glob.glob(str(CL_red_pattern))

print(
    "Checking for raw data files at:\n",
    f" ./{Path(CL_red_pattern).relative_to(PROJECT_ROOT)}\n"
)

if matching_files:
    print(f"✅ Found {len(matching_files)} raw data file(s) for CL_red channel")
    print(f"   Example: {Path(matching_files[0]).name}")
    
    # Show a few more examples if available
    if len(matching_files) > 1:
        for i, file_path in enumerate(matching_files[1:4], start=2):
            print(f"            {Path(file_path).name}")
        if len(matching_files) > 4:
            print(f"            ... and {len(matching_files) - 4} more")
else:
    print("⚠️  No raw data files found yet (configuration-only demo)")
    print("\n📋 This is expected when exploring the configuration system")
    print("   without having copied the microscopy data yet.")
    print("\n   Microscopy data should be placed at, e.g.:")
    print(f"     ./{Path(CL_red_pattern).parent.relative_to(PROJECT_ROOT)}/")
    print(f"   with filenames matching pattern: {Path(CL_red_pattern).name}")

Checking for raw data files at:
  ./data/training/physical/single_particle/2024-06-29_Dy100/raw/region_1/CL_SE/CL/red/593LP_NP_*

⚠️  No raw data files found yet (configuration-only demo)

📋 This is expected when exploring the configuration system
   without having copied the microscopy data yet.

   Microscopy data should be placed at, e.g.:
     ./data/training/physical/single_particle/2024-06-29_Dy100/raw/region_1/CL_SE/CL/red/
   with filenames matching pattern: 593LP_NP_*


## 8. Processed Data File Paths

Processed outputs follow a similar hierarchy but with method-specific subdirectories:

```
processed/
├── {method_name}/
│   └── {region_id}/
│       ├── CL_SE/
│       │   ├── CL/
│       │   │   ├── blue/
│       │   │   ├── green/
│       │   │   └── red/
│       │   └── SE/
│       └── SEM/
```

In [12]:
# Get processed file paths for summed_images method (uses already-validated configs)
method_name = 'summed_images'
processed_paths = config_loader.get_processed_filepaths(
    exp_id, 
    method_name=method_name, 
    region_id=region_id,
    validated_configs=validated_configs
)

print(f"Processed output paths for '{method_name}' method, {region_id}:\n")
for channel, path in processed_paths.items():
    print(f"  {channel:12s}: ./{Path(path).relative_to(PROJECT_ROOT)}")

Processed output paths for 'summed_images' method, region_1:

  CL_blue     : ./data/training/physical/single_particle/2024-06-29_Dy100/processed/summed_images/region_1/CL_SE/CL/blue/summed_blue_region_1.tif
  CL_green    : ./data/training/physical/single_particle/2024-06-29_Dy100/processed/summed_images/region_1/CL_SE/CL/green/summed_green_region_1.tif
  CL_red      : ./data/training/physical/single_particle/2024-06-29_Dy100/processed/summed_images/region_1/CL_SE/CL/red/summed_red_region_1.tif
  SE          : ./data/training/physical/single_particle/2024-06-29_Dy100/processed/summed_images/region_1/CL_SE/SE/summed_SE2_region_1.tif


## 9. Channel Information Queries

The system provides direct access to channel-specific configuration without manual navigation of nested dictionaries.

In [13]:
# Get information for the red CL channel (uses already-validated configs)
red_channel_info = config_loader.get_raw_channel_info(
    exp_id, 
    acq_type='CL_SE', 
    detector='CL', 
    channel_name='red',
    validated_configs=validated_configs
)

print("Red CL channel configuration:")
pprint(red_channel_info)

print(f"\nTarget dopant for red channel: {red_channel_info['target_dopant']}")
print(f"Filename prefix pattern: {red_channel_info['source_prefix']}")

Red CL channel configuration:
{'source_prefix': '593LP_NP_', 'target_dopant': 'Ho'}

Target dopant for red channel: Ho
Filename prefix pattern: 593LP_NP_


In [14]:
# Get processed output information for the same channel (uses already-validated configs)
red_processed_info = config_loader.get_processed_channel_info(
    exp_id, 
    method_name='summed_images',
    acq_type='CL_SE', 
    detector='CL', 
    channel_name='red',
    validated_configs=validated_configs
)

print("Red CL channel processed output configuration:")
pprint(red_processed_info)

Red CL channel processed output configuration:
{'filename_prefix': 'summed_red'}


## 10. Demonstration Recap

### Key Features Demonstrated

1. **Comprehensive validation**
   - Schema validation using Pydantic
   - Cross-reference validation between configuration files
   - Elemental fractions and mixture priors constraints

2. **Configuration merging**
   - Automatic merging from 3 YAML sources
   - Different configs for training vs. inference experiments

3. **Path generation**
   - Consistent directory hierarchy
   - Automatic template substitution
   - Support for multiple regions and processing methods

4. **Channel information queries**
   - Simplified API for accessing nested configurations with validation

### Performance Note

Notice that we validated configurations **only once** (Section 2) and then passed the validated objects to all subsequent functions via the `validated_configs` parameter. This avoids redundant validation and improves performance.

### Next Steps

This configuration system provides the foundation for:
- **Image preprocessing**: Automated processing of raw microscopy data (SEM, CL, SE)
- **Feature extraction**: Acquisition- and channel-specific feature computation
- **Model training**: Building classifiers using labeled training data with known compositions
- **Inference**: Applying trained models to classify individual nanoparticles within multi-particle mixtures (Bayesian methods in progress, transformer-based planned)

---

**For more details, see:**
- `README.md` - Complete architecture documentation
- `src/data/schemas.py` - Pydantic validation schemas
- `src/data/config_loader.py` - Configuration loading implementation
- `tests/` - Comprehensive test suite